#1. Setup

In [1]:
# Install dependencies
!pip -q install -U transformers accelerate safetensors

import torch
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, AutoModelForCausalLM


# --------------------------------------------------
# Baseline configuration
# --------------------------------------------------

MODEL_ID = "Qwen/Qwen2.5-1.5B"
DTYPE = torch.float16


print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("A GPU runtime is required for this notebook.")

gpu = torch.cuda.get_device_properties(0)

print("GPU:", gpu.name)
print(f"VRAM: {gpu.total_memory / 1024**3:.2f} GiB")


# --------------------------------------------------
# Reload the baseline model
# --------------------------------------------------

print("\nLoading:", MODEL_ID)
print("Using dtype:", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=DTYPE,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)

model.eval()


# --------------------------------------------------
# Model architecture
# --------------------------------------------------

config = model.config

num_layers = config.num_hidden_layers
num_attention_heads = config.num_attention_heads
num_kv_heads = getattr(
    config,
    "num_key_value_heads",
    num_attention_heads
)

hidden_size = config.hidden_size

head_dim = getattr(
    config,
    "head_dim",
    hidden_size // num_attention_heads
)

bytes_per_element = torch.tensor(
    [],
    dtype=DTYPE
).element_size()


print("\nModel loaded.")
print("Layers:", num_layers)
print("Attention heads:", num_attention_heads)
print("KV heads:", num_kv_heads)
print("Head dimension:", head_dim)
print("Bytes per FP16 element:", bytes_per_element)

print(
    f"GPU memory allocated: "
    f"{torch.cuda.memory_allocated() / 1024**3:.3f} GiB"
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 42.4 MB/s eta 0:00:00
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GiB

Loading: Qwen/Qwen2.5-1.5B
Using dtype: torch.float16


config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.23k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]


Model loaded.
Layers: 28
Attention heads: 12
KV heads: 2
Head dimension: 128
Bytes per FP16 element: 2
GPU memory allocated: 2.875 GiB


#2. Create the analysis KV cache

In [2]:
# --------------------------------------------------
# Create a 2048-token sequence for KV-cache analysis
# --------------------------------------------------

ANALYSIS_SEQ_LEN = 2048

text = (
    "Large language models generate text autoregressively. "
    "During generation, attention layers reuse previously computed "
    "key and value vectors through the KV cache. "
    "Efficient KV-cache management becomes increasingly important "
    "as the context length grows. "
) * 500

analysis_input_ids = tokenizer(
    text,
    return_tensors="pt",
    add_special_tokens=False
).input_ids[:, :ANALYSIS_SEQ_LEN]

analysis_input_ids = analysis_input_ids.to(model.device)

print("Input shape:", tuple(analysis_input_ids.shape))


# --------------------------------------------------
# Build the FP16 KV cache
# --------------------------------------------------

torch.cuda.reset_peak_memory_stats()

with torch.inference_mode():
    analysis_outputs = model(
        input_ids=analysis_input_ids,
        use_cache=True
    )

analysis_cache = analysis_outputs.past_key_values


# --------------------------------------------------
# Inspect the cache
# --------------------------------------------------

print("\nCache type:", type(analysis_cache))
print("Number of cache layers:", len(analysis_cache.layers))
print("Cached sequence length:", analysis_cache.get_seq_length())

key0 = analysis_cache.layers[0].keys
value0 = analysis_cache.layers[0].values

print("\nLayer 0 K shape:", tuple(key0.shape))
print("Layer 0 V shape:", tuple(value0.shape))
print("K dtype:", key0.dtype)
print("V dtype:", value0.dtype)


# --------------------------------------------------
# Measure total cache memory
# --------------------------------------------------

cache_bytes = 0

for layer in analysis_cache.layers:
    cache_bytes += layer.keys.numel() * layer.keys.element_size()
    cache_bytes += layer.values.numel() * layer.values.element_size()

cache_mib = cache_bytes / (1024 ** 2)

print(f"\nTotal FP16 KV-cache memory: {cache_mib:.2f} MiB")
print(
    f"Peak GPU memory during forward: "
    f"{torch.cuda.max_memory_allocated() / 1024**3:.3f} GiB"
)

Input shape: (1, 2048)

Cache type: <class 'transformers.cache_utils.DynamicCache'>
Number of cache layers: 28
Cached sequence length: 2048

Layer 0 K shape: (1, 2, 2048, 128)
Layer 0 V shape: (1, 2, 2048, 128)
K dtype: torch.float16
V dtype: torch.float16

Total FP16 KV-cache memory: 56.00 MiB
Peak GPU memory during forward: 3.525 GiB


A real 2048-token KV cache that we'll use in the next cells for paging. We use 2048 because it is large enough to contain many pages

#3. Page size selection

In [3]:
# --------------------------------------------------
# Compare candidate KV-cache page sizes
# --------------------------------------------------

candidate_page_sizes = [8, 16, 32, 64]

rows = []

# We already measured 28 KiB of FP16 KV cache per token
kv_bytes_per_token = (
    2                       # K + V
    * num_layers
    * num_kv_heads
    * head_dim
    * bytes_per_element
)

for page_size in candidate_page_sizes:

    num_pages = (
        ANALYSIS_SEQ_LEN + page_size - 1
    ) // page_size

    page_bytes = page_size * kv_bytes_per_token

    rows.append({
        "Page Size (tokens)": page_size,
        "Pages @ 2048 tokens": num_pages,
        "FP16 Memory / Page (KiB)": page_bytes / 1024
    })

page_size_df = pd.DataFrame(rows)

display(page_size_df)


# Initial project default
PAGE_SIZE = 16

num_pages = (
    ANALYSIS_SEQ_LEN + PAGE_SIZE - 1
) // PAGE_SIZE

print("Selected page size:", PAGE_SIZE, "tokens")
print("Number of logical pages:", num_pages)

,Page Size (tokens),Pages @ 2048 tokens,FP16 Memory / Page (KiB)
0,8,256,224.0
1,16,128,448.0
2,32,64,896.0
3,64,32,1792.0


Selected page size: 16 tokens
Number of logical pages: 128


#4. Split K/V tensors into pages

In [4]:
# --------------------------------------------------
# Split one layer's K/V cache into token pages
# --------------------------------------------------

def split_tensor_into_pages(tensor, page_size):
    """
    Split a KV tensor along the sequence dimension.

    Expected tensor shape:
        [batch, kv_heads, sequence_length, head_dim]

    Returns:
        list of page tensors
    """
    seq_len = tensor.shape[2]

    pages = []

    for start in range(0, seq_len, page_size):
        end = min(start + page_size, seq_len)

        page = tensor[:, :, start:end, :]

        pages.append(page)

    return pages


# --------------------------------------------------
# Test on Layer 0
# --------------------------------------------------

layer_idx = 0

layer0_key = analysis_cache.layers[layer_idx].keys
layer0_value = analysis_cache.layers[layer_idx].values

key_pages = split_tensor_into_pages(
    layer0_key,
    PAGE_SIZE
)

value_pages = split_tensor_into_pages(
    layer0_value,
    PAGE_SIZE
)


print("Original K shape:", tuple(layer0_key.shape))
print("Original V shape:", tuple(layer0_value.shape))

print("\nNumber of K pages:", len(key_pages))
print("Number of V pages:", len(value_pages))

print("\nPage 0 K shape:", tuple(key_pages[0].shape))
print("Page 0 V shape:", tuple(value_pages[0].shape))

print("Page 1 K shape:", tuple(key_pages[1].shape))
print("Last page K shape:", tuple(key_pages[-1].shape))

Original K shape: (1, 2, 2048, 128)
Original V shape: (1, 2, 2048, 128)

Number of K pages: 128
Number of V pages: 128

Page 0 K shape: (1, 2, 16, 128)
Page 0 V shape: (1, 2, 16, 128)
Page 1 K shape: (1, 2, 16, 128)
Last page K shape: (1, 2, 16, 128)


In [5]:
for page_idx in [0, 1, 2, 127]:
    start_token = page_idx * PAGE_SIZE
    end_token = min(
        start_token + PAGE_SIZE,
        ANALYSIS_SEQ_LEN
    )

    print(
        f"Page {page_idx:3d}: "
        f"tokens {start_token:4d} to {end_token - 1:4d}"
    )

Page   0: tokens    0 to   15
Page   1: tokens   16 to   31
Page   2: tokens   32 to   47
Page 127: tokens 2032 to 2047


### Page Representation

The tensor splitting above is performed **for one transformer layer at a time**. With a page size of 16 tokens, each layer contains:

$$K_{\text{page}}, V_{\text{page}} \in \mathbb{R}^{1\times2\times16\times128}$$

However, a **logical page** in this project represents the same token range across the **entire model**. For example:

$$\text{Page 0 = tokens 0--15 across K and V of all 28 layers}$$

Thus, each logical page groups the KV-cache information for a fixed range of tokens across all transformer layers.

#5. Build page metadata

In [6]:
# --------------------------------------------------
# Build metadata for every logical KV-cache page
# --------------------------------------------------

page_metadata = []

for page_idx in range(num_pages):

    start_token = page_idx * PAGE_SIZE
    end_token = min(
        start_token + PAGE_SIZE,
        ANALYSIS_SEQ_LEN
    )

    num_tokens = end_token - start_token

    # How far this page is from the end of the sequence
    age_tokens = ANALYSIS_SEQ_LEN - end_token

    # FP16 memory occupied by this logical page
    # across K + V and all transformer layers
    memory_bytes = num_tokens * kv_bytes_per_token

    page_metadata.append({
        "page_index": page_idx,
        "start_token": start_token,
        "end_token": end_token - 1,
        "num_tokens": num_tokens,

        # Positional information
        "age_tokens": age_tokens,
        "is_prefix_page": (page_idx == 0),

        # Current storage state
        "precision": "FP16",
        "memory_kib": memory_bytes / 1024,
    })


page_df = pd.DataFrame(page_metadata)

display(page_df.head())
print("...")
display(page_df.tail())

print("\nNumber of pages:", len(page_df))
print(
    "Total page memory:",
    f"{page_df['memory_kib'].sum() / 1024:.2f} MiB"
)

,page_index,start_token,end_token,num_tokens,age_tokens,is_prefix_page,precision,memory_kib
0,0,0,15,16,2032,True,FP16,448.0
1,1,16,31,16,2016,False,FP16,448.0
2,2,32,47,16,2000,False,FP16,448.0
3,3,48,63,16,1984,False,FP16,448.0
4,4,64,79,16,1968,False,FP16,448.0


...


,page_index,start_token,end_token,num_tokens,age_tokens,is_prefix_page,precision,memory_kib
123,123,1968,1983,16,64,False,FP16,448.0
124,124,1984,1999,16,48,False,FP16,448.0
125,125,2000,2015,16,32,False,FP16,448.0
126,126,2016,2031,16,16,False,FP16,448.0
127,127,2032,2047,16,0,False,FP16,448.0



Number of pages: 128
Total page memory: 56.00 MiB


The important validation is:

128×448 KiB=56 MiB

which exactly matches the original FP16 KV-cache memory from Cell 2. So the metadata representation accounts for the entire cache correctly.

#6. Verify paging is lossless

In [7]:
# --------------------------------------------------
# Reconstruct tensors from pages
# --------------------------------------------------

reconstructed_key = torch.cat(
    key_pages,
    dim=2       # sequence dimension
)

reconstructed_value = torch.cat(
    value_pages,
    dim=2
)


print("Original K shape:     ", tuple(layer0_key.shape))
print("Reconstructed K shape:", tuple(reconstructed_key.shape))

print("\nOriginal V shape:     ", tuple(layer0_value.shape))
print("Reconstructed V shape:", tuple(reconstructed_value.shape))


# --------------------------------------------------
# Verify exact equality
# --------------------------------------------------

key_equal = torch.equal(
    layer0_key,
    reconstructed_key
)

value_equal = torch.equal(
    layer0_value,
    reconstructed_value
)

key_max_diff = (
    layer0_key.float() -
    reconstructed_key.float()
).abs().max().item()

value_max_diff = (
    layer0_value.float() -
    reconstructed_value.float()
).abs().max().item()


print("\nExact K equality:", key_equal)
print("Exact V equality:", value_equal)

print("Maximum K difference:", key_max_diff)
print("Maximum V difference:", value_max_diff)

Original K shape:      (1, 2, 2048, 128)
Reconstructed K shape: (1, 2, 2048, 128)

Original V shape:      (1, 2, 2048, 128)
Reconstructed V shape: (1, 2, 2048, 128)

Exact K equality: True
Exact V equality: True
Maximum K difference: 0.0
Maximum V difference: 0.0


Lossless Page Reconstruction

The KV tensors were split into 16-token pages and reconstructed by concatenating the pages along the sequence dimension. The reconstructed Key and Value tensors were exactly equal to the original tensors, with a maximum numerical difference of 0. Therefore, the page representation itself introduces no approximation error. Any error observed later can be attributed to quantization or precision assignment rather than paging.

#7. Final page-level summary

In [8]:
# --------------------------------------------------
# Final Notebook 02 summary
# --------------------------------------------------

page_summary = {
    "model": MODEL_ID,
    "sequence_length": ANALYSIS_SEQ_LEN,
    "page_size_tokens": PAGE_SIZE,
    "num_pages": num_pages,

    "num_layers": num_layers,
    "kv_heads": num_kv_heads,
    "head_dim": head_dim,

    "fp16_kv_cache_mib": cache_mib,
    "fp16_memory_per_page_kib": (
        PAGE_SIZE * kv_bytes_per_token / 1024
    ),

    "first_page_token_range": [
        int(page_df.iloc[0]["start_token"]),
        int(page_df.iloc[0]["end_token"])
    ],

    "last_page_token_range": [
        int(page_df.iloc[-1]["start_token"]),
        int(page_df.iloc[-1]["end_token"])
    ],

    "paging_lossless": bool(
        key_equal and value_equal
    )
}


print("KV Page Representation Summary")
print("------------------------------")

for key, value in page_summary.items():
    print(f"{key}: {value}")


# --------------------------------------------------
# Save metadata and summary
# --------------------------------------------------

import json

page_df.to_csv(
    "kv_page_metadata.csv",
    index=False
)

with open(
    "kv_page_summary.json",
    "w"
) as f:
    json.dump(
        page_summary,
        f,
        indent=4
    )


print("\nSaved:")
print("  kv_page_metadata.csv")
print("  kv_page_summary.json")

KV Page Representation Summary
------------------------------
model: Qwen/Qwen2.5-1.5B
sequence_length: 2048
page_size_tokens: 16
num_pages: 128
num_layers: 28
kv_heads: 2
head_dim: 128
fp16_kv_cache_mib: 56.0
fp16_memory_per_page_kib: 448.0
first_page_token_range: [0, 15]
last_page_token_range: [2032, 2047]
paging_lossless: True

Saved:
  kv_page_metadata.csv
  kv_page_summary.json
